# Validación de datos generados

**Proyecto:** Piquillo BI - AgroPiura Conservas S.A.C.

Este notebook valida que los CSVs generados por `orchestrator.py` cumplen con los criterios de calidad:

1. Volumetría coherente con lo configurado
2. Sin nulos en claves primarias/foráneas
3. KPIs de negocio en rangos realistas
4. Estacionalidad respetada
5. SCD2 bien armado (un solo `EsActual=1` por entidad)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../../data/raw")

def leer(nombre):
    return pd.read_csv(DATA / f"{nombre}.csv", encoding="utf-8-sig")

dim_fecha = leer("DimFecha")
dim_prod = leer("DimProductor")
dim_fundo = leer("DimFundo")
dim_parcela = leer("DimParcela")
dim_cliente = leer("DimCliente")
dim_precio = leer("DimPrecioRefSUNAT")
fact_cos = leer("FactCosecha")
fact_proc = leer("FactProceso")
fact_desp = leer("FactDespacho")

print("Archivos cargados")
for nombre, df in [("DimFecha", dim_fecha), ("DimProductor", dim_prod),
                   ("DimFundo", dim_fundo), ("DimParcela", dim_parcela),
                   ("DimCliente", dim_cliente), ("DimPrecioRefSUNAT", dim_precio),
                   ("FactCosecha", fact_cos), ("FactProceso", fact_proc),
                   ("FactDespacho", fact_desp)]:
    print(f"  {nombre:25s} {len(df):>7,} filas")

## 1. Volumetría

In [ ]:
total_hechos = len(fact_cos) + len(fact_proc) + len(fact_desp)
print(f"Total filas en hechos: {total_hechos:,}")
print(f"Objetivo: 18,000 - 30,000")
assert 15000 <= total_hechos <= 35000, "Volumetría fuera de rango"
print("OK volumetría")

## 2. Integridad SCD2 - DimProductor

In [ ]:
# Cada ProductorID debe tener exactamente 1 fila con EsActual=1
actuales_por_prod = dim_prod[dim_prod["EsActual"] == 1].groupby("ProductorID").size()
print("Productores con EsActual=1:", (actuales_por_prod == 1).sum())
print("Productores con problema:", (actuales_por_prod != 1).sum())
assert (actuales_por_prod == 1).all(), "SCD2 mal armado"

# Versiones por productor
versiones = dim_prod.groupby("ProductorID").size().value_counts().sort_index()
print("\nDistribución de versiones por productor:")
print(versiones)

## 3. KPIs agrícolas en rango realista

In [ ]:
# Rendimiento por hectárea por campaña
df = fact_cos.merge(dim_parcela[["ParcelaID", "AreaHa"]], on="ParcelaID")
rend = df.groupby("Campania").apply(
    lambda x: x["KgCosechados"].sum() / x["AreaHa"].sum()
)
print("Rendimiento Kg/Ha por campaña:")
print(rend.round(0))
print("\nRango esperado: 14,000 - 28,000 kg/ha")

# Tasa rechazo campo
tasa_rec = fact_cos["KgRechazadosCampo"].sum() / fact_cos["KgCosechados"].sum()
print(f"\nTasa rechazo campo global: {tasa_rec:.2%} (esperado ~6%)")

## 4. KPIs industriales

In [ ]:
rend_proc = fact_proc["KgProductoTerminado"].sum() / fact_proc["KgIngresoMP"].sum()
merma = fact_proc["KgMermaProceso"].sum() / fact_proc["KgIngresoMP"].sum()
rechazo = fact_proc["KgRechazoCalidad"].sum() / fact_proc["KgIngresoMP"].sum()

print(f"Rendimiento proceso global:  {rend_proc:.2%} (esperado 60-72%)")
print(f"Tasa merma proceso:           {merma:.2%}")
print(f"Tasa rechazo calidad:         {rechazo:.2%}")

## 5. KPIs comerciales

In [ ]:
fob_avg = fact_desp["ValorFOB_USD"].sum() / fact_desp["KgNetosExportados"].sum()
print(f"FOB promedio USD/Kg: {fob_avg:.2f} (esperado 2.80 - 4.20)")

# DIFOT
difot = (fact_desp["EstadoDespacho"] == "OnTime").mean()
print(f"DIFOT %: {difot:.1%} (esperado ~85%)")

# Distribución de destinos
print("\nTop destinos por valor FOB:")
top = fact_desp.merge(leer("DimDestino")[["DestinoID", "Pais"]], on="DestinoID")
print(top.groupby("Pais")["ValorFOB_USD"].sum().sort_values(ascending=False).head(10))

## 6. Estacionalidad de cosecha

In [ ]:
fact_cos["FechaCosecha"] = pd.to_datetime(fact_cos["FechaCosecha"])
por_mes = fact_cos.groupby(fact_cos["FechaCosecha"].dt.month)["KgCosechados"].sum()
por_mes_pct = (por_mes / por_mes.sum() * 100).round(1)
print("Distribución mensual de kg cosechados (%):")
print(por_mes_pct)
print("\nMáximo esperado en Agosto (peak campaña principal)")

## 7. Integridad referencial

In [ ]:
# FK FactCosecha -> DimParcela
huerfanos = ~fact_cos["ParcelaID"].isin(dim_parcela["ParcelaID"])
print(f"FactCosecha huérfanos en DimParcela: {huerfanos.sum()}")

# FK FactDespacho -> DimCliente
huerfanos = ~fact_desp["ClienteID"].isin(dim_cliente["ClienteID"])
print(f"FactDespacho huérfanos en DimCliente: {huerfanos.sum()}")

# Trazabilidad: FactDespacho.LoteProcesoID debe existir en FactProceso
huerfanos = ~fact_desp["LoteProcesoID"].isin(fact_proc["LoteProcesoID"])
print(f"FactDespacho huérfanos en FactProceso: {huerfanos.sum()}")